In [2]:
%pip install --upgrade pip
%pip install -q diffusers transformers accelerate peft bitsandbytes
%pip install xformers==0.0.27.post2
%pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
from datasets import load_dataset
from torchvision import transforms
from torch. utils.data import DataLoader
from diffusers import StableDiffusionXLPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
import os

# ================== 설정 ==================
DATASET_NAME = "lambdalabs/naruto-blip-captions"
MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0"
OUTPUT_DIR = "./sdxl-lora-output"
RESOLUTION = 512
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 3

device = "cuda" if torch.cuda.is_available() else "cpu"

# ================== 모델 로드 ==================
print("모델 로딩 중...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch. float16,
    use_safetensors=True,
    variant="fp16"
)

# ================== LoRA 설정 및 적용 ==================
print("LoRA 설정 적용 중...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "to_q", "to_k", "to_v", "to_out.0",
    ],
    lora_dropout=0.05,
)

unet = pipe.unet
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

# ================== 메모리 최적화 ==================
print("메모리 최적화 설정 중...")

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers 활성화 완료")
except Exception as e:
    print(f"xformers 사용 불가: {e}")
    pipe.enable_attention_slicing()

pipe.enable_vae_slicing()
torch.cuda.empty_cache()

# ================== 모델 dtype 통일 ==================
print("모델 dtype 설정 중...")

pipe.vae.to(device, dtype=torch.float16)
pipe.text_encoder.to(device, dtype=torch. float16)
pipe.text_encoder_2.to(device, dtype=torch.float16)
unet.to(device)

pipe.vae.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)
pipe.text_encoder_2.requires_grad_(False)

# ================== SDXL용 헬퍼 함수 ==================
def compute_time_ids(original_size, crops_coords_top_left, target_size, device, dtype):
    """SDXL에 필요한 time_ids 생성"""
    add_time_ids = list(original_size + crops_coords_top_left + target_size)
    add_time_ids = torch.tensor([add_time_ids], dtype=dtype, device=device)
    return add_time_ids

def encode_prompt_sdxl(pipe, prompt, device):
    """SDXL용 듀얼 텍스트 인코더 처리"""
    # Text Encoder 1
    text_inputs = pipe.tokenizer(
        prompt,
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt"
    )
    text_input_ids = text_inputs.input_ids. to(device)
    
    with torch.no_grad():
        prompt_embeds = pipe.text_encoder(
            text_input_ids,
            output_hidden_states=True
        )
        pooled_prompt_embeds = prompt_embeds[0]
        prompt_embeds = prompt_embeds.hidden_states[-2]
    
    # Text Encoder 2
    text_inputs_2 = pipe.tokenizer_2(
        prompt,
        padding="max_length",
        max_length=pipe.tokenizer_2.model_max_length,
        truncation=True,
        return_tensors="pt"
    )
    text_input_ids_2 = text_inputs_2.input_ids.to(device)
    
    with torch.no_grad():
        prompt_embeds_2 = pipe.text_encoder_2(
            text_input_ids_2,
            output_hidden_states=True
        )
        pooled_prompt_embeds_2 = prompt_embeds_2[0]
        prompt_embeds_2 = prompt_embeds_2.hidden_states[-2]
    
    # 두 임베딩 결합
    prompt_embeds = torch.concat([prompt_embeds, prompt_embeds_2], dim=-1)
    
    return prompt_embeds, pooled_prompt_embeds_2

# ================== 데이터셋 로드 ==================
print("데이터셋 로딩 중...")
dataset = load_dataset(DATASET_NAME, split='train[:100]')
print(f'original dataset:  {dataset.column_names}')

transform = transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def preprocess(example):
    image = example['image'].convert('RGB')
    return {
        'pixel_values': transform(image),
        'caption': example['text']
    }

dataset = dataset. map(preprocess, remove_columns=dataset.column_names)
dataset.set_format(type='torch', columns=['pixel_values', 'caption'])
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ================== 학습 설정 ==================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=LEARNING_RATE
)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_NAME, subfolder='scheduler')
scaler = torch.cuda. amp.GradScaler()

# ================== 학습 루프 ==================
print("LoRA 학습 시작!")
unet.train()

for epoch in range(EPOCHS):
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    total_loss = 0

    for step, batch in enumerate(progress_bar):

        with torch.cuda.amp.autocast(dtype=torch. float16):
            pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)

            # Latent 인코딩
            with torch.no_grad():
                latents = pipe.vae.encode(pixel_values).latent_dist.sample()
                latents = latents * pipe.vae.config. scaling_factor

            # 노이즈 추가
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config. num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # 캡션 처리
            captions = batch["caption"]
            if isinstance(captions, torch.Tensor):
                captions = [str(c) for c in captions]
            elif isinstance(captions, str):
                captions = [captions]
            else:
                captions = list(captions)

            # SDXL 텍스트 임베딩 (듀얼 인코더)
            prompt_embeds, pooled_prompt_embeds = encode_prompt_sdxl(
                pipe, captions, device
            )
            prompt_embeds = prompt_embeds.to(dtype=torch.float16)
            pooled_prompt_embeds = pooled_prompt_embeds. to(dtype=torch.float16)

            # SDXL time_ids 생성
            original_size = (RESOLUTION, RESOLUTION)
            crops_coords_top_left = (0, 0)
            target_size = (RESOLUTION, RESOLUTION)
            
            add_time_ids = compute_time_ids(
                original_size,
                crops_coords_top_left,
                target_size,
                device,
                torch. float16
            )
            # 배치 크기에 맞게 확장
            add_time_ids = add_time_ids.repeat(latents.shape[0], 1)

            # SDXL UNet 예측 (added_cond_kwargs 포함)
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=prompt_embeds,
                added_cond_kwargs={
                    "text_embeds": pooled_prompt_embeds,
                    "time_ids": add_time_ids
                },
                return_dict=False
            )[0]

            # Loss 계산
            loss = torch.nn.functional.mse_loss(noise_pred, noise)
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} 완료 - 평균 Loss: {avg_loss:. 4f}")

# ================== LoRA 가중치 저장 ==================
print("LoRA 가중치 저장 중...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
unet.save_pretrained(OUTPUT_DIR)
print(f"저장 완료: {OUTPUT_DIR}")

print("LoRA 학습 완료!")

모델 로딩 중...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

LoRA 설정 적용 중...
trainable params: 11,612,160 || all params: 2,579,075,844 || trainable%: 0.4502
메모리 최적화 설정 중...
xformers 활성화 완료
모델 dtype 설정 중...
데이터셋 로딩 중...


Repo card metadata block was not found. Setting CardData to empty.


original dataset:  ['image', 'text']


/tmp/ipykernel_158/4249683587.py:150: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda. amp.GradScaler()


LoRA 학습 시작!


Epoch 1/3:   0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_158/4249683587.py:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch. float16):
Epoch 1/3:  44%|████▍     | 44/100 [00:30<00:39,  1.43it/s, loss=nan]